## 1. CARGA DE ARCHIVOS .CSV

In [14]:
import pandas as pd
from pathlib import Path

# Desde 'notebooks', subimos un nivel con '..' y entramos a 'data/raw'
DATA_PATH = Path("../data/raw")

orders = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
sellers = pd.read_csv(DATA_PATH / 'olist_sellers_dataset.csv')
payments = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')
category_translation = pd.read_csv(DATA_PATH / 'product_category_name_translation.csv')

print("¡Archivos cargados correctamente!")
print("Columnas de órdenes:", orders.columns)


¡Archivos cargados correctamente!
Columnas de órdenes: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')


## 2. CREACIÓN DE TABLAS NORMALIZADAS (MODELO RACIONAL)

In [24]:

# a) Tabla Ciudades (Ciudades unificadas de Clientes y Vendedores)
ciudades_unicas = pd.concat([
    customers['customer_city'], 
    sellers['seller_city']
]).unique()

Ciudades = pd.DataFrame(ciudades_unicas, columns=['Ciudad'])
Ciudades['CiudadID'] = Ciudades.index + 1
Ciudades = Ciudades[['CiudadID', 'Ciudad']]

Ciudades.head()


,CiudadID,Ciudad
0,1,franca
1,2,sao bernardo do campo
2,3,sao paulo
3,4,mogi das cruzes
4,5,campinas


In [25]:
# b) Tabla Vendedores
Vendedores = sellers[['seller_id', 'seller_city', 'seller_state']].drop_duplicates().reset_index(drop=True)
Vendedores = Vendedores.merge(Ciudades, left_on='seller_city', right_on='Ciudad', how='left')
Vendedores['VendedorID'] = Vendedores.index + 1
Vendedores = Vendedores[['VendedorID', 'seller_id', 'CiudadID', 'seller_state']]
Vendedores.head()

,VendedorID,seller_id,CiudadID,seller_state
0,1,3442f8959a84dea7ee197c632cb2df15,5,SP
1,2,d1b65fc7debc3361ea86b5f14c68d2e2,4120,SP
2,3,ce3ad9de960102d0677a81f5d0bb7b2d,11,RJ
3,4,c0f3eea2e14555b6faeea3dd58c1b1c3,3,SP
4,5,51a04a8a6bdcb23deccc82b0b80742cf,80,SP


In [26]:
# c) Tabla Clientes
Clientes = customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates().reset_index(drop=True)
Clientes = Clientes.merge(Ciudades, left_on='customer_city', right_on='Ciudad', how='left')
Clientes['ClienteID'] = Clientes.index + 1
Clientes = Clientes[['ClienteID', 'customer_id', 'customer_unique_id', 'CiudadID', 'customer_state']]
Clientes.head()

,ClienteID,customer_id,customer_unique_id,CiudadID,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,1,SP
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,2,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,3,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,4,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,5,SP


In [27]:


# d) Tabla Métodos de Pago
MetodosPago = pd.DataFrame(payments['payment_type'].unique(), columns=['MetodoPago'])
MetodosPago['MetodoPagoID'] = MetodosPago.index + 1
MetodosPago = MetodosPago[['MetodoPagoID', 'MetodoPago']]
MetodosPago.head()


,MetodoPagoID,MetodoPago
0,1,credit_card
1,2,boleto
2,3,voucher
3,4,debit_card
4,5,not_defined


In [28]:


# e) Tabla Productos (Catálogo traduciendo categoría al inglés)
Productos = products.merge(category_translation, on='product_category_name', how='left')
Productos = Productos[['product_id', 'product_category_name_english', 'product_weight_g']].drop_duplicates().reset_index(drop=True)
Productos['ProductoID'] = Productos.index + 1
Productos = Productos[['ProductoID', 'product_id', 'product_category_name_english', 'product_weight_g']]
Productos.head()


,ProductoID,product_id,product_category_name_english,product_weight_g
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,225.0
1,2,3aa071139cb16b67ca9e5dea641aaa2f,art,1000.0
2,3,96bd76ec8810374ed1b65e291975717f,sports_leisure,154.0
3,4,cef67bcfe19066a932b7673e239eb23d,baby,371.0
4,5,9dc1a7de274444849c219cff195d0b71,housewares,625.0


In [29]:


# f) Tabla Facturas (Nivel Orden / Transacción)
# Convertir a datetime y descomponer fecha como en tu proyecto
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['AñoVenta'] = orders['order_purchase_timestamp'].dt.year
orders['MesVenta'] = orders['order_purchase_timestamp'].dt.month
orders['DíaVenta'] = orders['order_purchase_timestamp'].dt.day
orders['HoraVenta'] = orders['order_purchase_timestamp'].dt.strftime('%H:%M:%S')

Facturas = orders[['order_id', 'customer_id', 'order_status', 'AñoVenta', 'MesVenta', 'DíaVenta', 'HoraVenta']].drop_duplicates().reset_index(drop=True)
Facturas = Facturas.merge(Clientes[['ClienteID', 'customer_id']], on='customer_id', how='left')
Facturas['VentaID'] = Facturas.index + 1
Facturas = Facturas[['VentaID', 'order_id', 'ClienteID', 'AñoVenta', 'MesVenta', 'DíaVenta', 'HoraVenta', 'order_status']]
Facturas.head()


,VentaID,order_id,ClienteID,AñoVenta,MesVenta,DíaVenta,HoraVenta,order_status
0,1,e481f51cbdc54678b7cc49136f2d6af7,70297,2017,10,2,10:56:33,delivered
1,2,53cdb2fc8bc7dce0b6741e2150273451,77028,2018,7,24,20:41:37,delivered
2,3,47770eb9100c2d0c44946d9cf07ec65d,555,2018,8,8,08:38:49,delivered
3,4,949d5b44dbf5de918fe9c16f97b45f8a,61082,2017,11,18,19:28:06,delivered
4,5,ad21c59c0840e6cb83a9ceb5573f8159,67264,2018,2,13,21:18:39,delivered


In [30]:


# g) Tabla Detalle de Facturas (Nivel Ítem / Producto)
DetalleFacturas = items.merge(Facturas[['VentaID', 'order_id']], on='order_id', how='left')
DetalleFacturas = DetalleFacturas.merge(Productos[['ProductoID', 'product_id']], on='product_id', how='left')
DetalleFacturas = DetalleFacturas.merge(Vendedores[['VendedorID', 'seller_id']], on='seller_id', how='left')

DetalleFacturas = DetalleFacturas.rename(columns={'price': 'Precio', 'freight_value': 'Flete'})
DetalleFacturas['Cantidad'] = 1  # Cada registro representa 1 ítem vendido
DetalleFacturas['Subtotal'] = DetalleFacturas['Precio'] + DetalleFacturas['Flete']

DetalleFacturas['DetalleID'] = DetalleFacturas.index + 1
DetalleFacturas = DetalleFacturas[['DetalleID', 'VentaID', 'ProductoID', 'VendedorID', 'Cantidad', 'Precio', 'Flete', 'Subtotal']]
DetalleFacturas.head()


,DetalleID,VentaID,ProductoID,VendedorID,Cantidad,Precio,Flete,Subtotal
0,1,85268,25866,514,1,58.90,13.29,72.19
1,2,71854,27231,472,1,239.90,19.93,259.83
2,3,6299,22625,1825,1,199.00,17.87,216.87
3,4,22551,15404,2024,1,12.99,12.79,25.78
4,5,5248,8863,1598,1,199.90,18.14,218.04


## 3. Análisis Exploratorio de Datos (EDA)

In [31]:
print("\n--- EDA: Frecuencia de estados de órdenes ---")
print(Facturas['order_status'].value_counts())

# Unir Detalle con Productos para métricas agrupadas (igual que en tu notebook)
df_ventas = DetalleFacturas.merge(Productos, on="ProductoID")

print("\n--- Categorías más vendidas ---")
categorias_mas_vendidas = df_ventas.groupby("product_category_name_english")["Cantidad"].sum().sort_values(ascending=False)
print(categorias_mas_vendidas.head(10))



--- EDA: Frecuencia de estados de órdenes ---
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

--- Categorías más vendidas ---
product_category_name_english
bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235
Name: Cantidad, dtype: int64


## 4. Exportación consolidado a Excel

In [ ]:
# 1. Lista con DataFrames reales
dataframes = [
    Ciudades,
    Vendedores,
    Clientes,
    MetodosPago,
    Productos,
    Facturas,
    DetalleFacturas
]


# 2. Nombres de las hojas en Excel
nombre_hojas = [
    "Ciudades",
    "Vendedores",
    "Clientes",
    "MetodosPago",
    "Productos",
    "Facturas",
    "DetalleFacturas"
]

# 3. Limpiamos valores nulos de forma segura
for df_temp in dataframes:
    df_temp.fillna(' ', inplace=True)

# 4. Ruta del archivo final
ruta_archivo = 'modeloVentasOlist.xlsx'

# 5. Exportación multi-hoja
with pd.ExcelWriter(ruta_archivo) as writer:
    for nombre_hoja, df_temp in zip(nombre_hojas, dataframes):
        df_temp.to_excel(writer, sheet_name=nombre_hoja, index=False)

print("\n¡Tablas exportadas con éxito total a modeloVentasOlist.xlsx!")